In [0]:
%sql
use dw01.gold;
create or replace table t_customers as
SELECT * FROM dw01.silver.t_customers;

SELECT * FROM dw01.gold.t_customers

In [0]:
%sql
use dw01.gold;

MERGE INTO t_customer AS target
USING dw01.silver.t_customers AS source
ON target.customer_id = source.customer_id
WHEN MATCHED AND (
       target.Customer_Name      <> source.Customer_Name
    OR target.Street             <> source.Street
    OR target.City               <> source.City
    OR target.State              <> source.State
    OR target.Customer_Type      <> source.Customer_Type
    OR target.Account_Manager    <> source.Account_Manager
) THEN
    UPDATE SET
        Customer_Name    = source.Customer_Name,
        Street           = source.Street,
        City             = source.City,
        State            = source.State,
        Customer_Type    = source.Customer_Type,
        Account_Manager  = source.Account_Manager
WHEN NOT MATCHED THEN
    INSERT (customer_id, Customer_Name, Street, City, State, Customer_Type, Account_Manager)
    VALUES (source.customer_id, source.Customer_Name, source.Street, source.City, source.State, source.Customer_Type, source.Account_Manager);


In [0]:
%sql
use dw01.gold;
create or replace table t_products as
SELECT * FROM dw01.silver.t_products;

SELECT * FROM dw01.gold.t_products

In [0]:
%sql
USE dw01.gold;

MERGE INTO t_products AS target
USING dw01.silver.t_products AS source
ON target.product_id = source.product_id
WHEN MATCHED AND (
       target.Product_Name      <> source.Product_Name
    OR target.Product_Category  <> source.Product_Category
    OR target.Product_Container <> source.Product_Container
) THEN
    UPDATE SET
        Product_Name      = source.Product_Name,
        Product_Category  = source.Product_Category,
        Product_Container = source.Product_Container
WHEN NOT MATCHED THEN
    INSERT (product_id, Product_Name, Product_Category, Product_Container)
    VALUES (source.product_id, source.Product_Name, source.Product_Category, source.Product_Container);


In [0]:
%sql
use dw01.gold;
create or replace table t_shippingdetails as
SELECT * FROM dw01.silver.t_shippingdetails;

SELECT * FROM dw01.gold.t_shippingdetails



In [0]:
%sql
USE dw01.gold;

MERGE INTO t_shippingdetails AS target
USING dw01.silver.t_shippingdetails AS source
ON target.shipping_id = source.shipping_id
WHEN MATCHED AND target.Ship_Mode <> source.Ship_Mode THEN
    UPDATE SET
        Ship_Mode = source.Ship_Mode
WHEN NOT MATCHED THEN
    INSERT (shipping_id, Ship_Mode)
    VALUES (source.shipping_id, source.Ship_Mode);


In [0]:
%sql
use dw01.gold;
create or replace table t_dates as
SELECT * FROM dw01.silver.t_dates;

SELECT * FROM dw01.gold.t_dates;

In [0]:
%sql
use dw01.gold;

MERGE INTO t_dates AS target
USING dw01.silver.t_dates AS source
ON target.date_id = source.date_id  -- match on primary/natural key
WHEN NOT MATCHED THEN
  INSERT *

In [0]:
%sql

use dw01.gold;
CREATE OR REPLACE TABLE t_orders
WITH unique_customers AS (
    SELECT *
    FROM (
        SELECT *,
               ROW_NUMBER() OVER (
                   PARTITION BY Customer_Name, City, State, Customer_Type, Account_Manager
                   ORDER BY customer_id
               ) AS rn
        FROM dw01.silver.t_customers
    ) t
    WHERE rn = 1
),
unique_products AS (
    SELECT *
    FROM (
        SELECT *,
               ROW_NUMBER() OVER (
                   PARTITION BY Product_Name, Product_Category, Product_Container
                   ORDER BY product_id
               ) AS rn
        FROM dw01.silver.t_products
    ) t
    WHERE rn = 1
),
unique_shipping AS (
    SELECT *
    FROM (
        SELECT *,
               ROW_NUMBER() OVER (
                   PARTITION BY Ship_Mode
                   ORDER BY shipping_id
               ) AS rn
        FROM dw01.silver.t_shippingdetails
    ) t
    WHERE rn = 1
),
main AS (
    SELECT
        o.order_id,
        o.Order_No,
        c.customer_id,
        p.product_id,
        s.shipping_id,
        dt_order.date_id AS order_date_id,
        dt_ship.date_id AS ship_date_id,
        o.Customer_Name,
        o.Address,
        o.City,
        o.State,
        o.Customer_Type,
        o.Account_Manager,
        o.Order_Priority,
        o.Product_Name,
        p.Product_Category,
        p.Product_Container,
        o.Ship_Mode,
        o.Ship_Date,
        o.Cost_Price,
        o.Retail_Price,
        o.Profit_Margin,
        o.Order_Quantity,
        o.Sub_Total,
        o.Discount_Amount,
        o.Order_Total,
        o.Shipping_Cost,
        o.Total
    FROM dw01.silver.t_orders o
    LEFT JOIN unique_customers c
        ON o.Customer_Name = c.Customer_Name
       AND o.City = c.City
       AND o.State = c.State
       AND o.Customer_Type = c.Customer_Type
       AND o.Account_Manager = c.Account_Manager
    LEFT JOIN unique_products p
        ON o.Product_Name = p.Product_Name
       AND o.Product_Category = p.Product_Category
       AND o.Product_Container = p.Product_Container
    LEFT JOIN unique_shipping s
        ON o.Ship_Mode = s.Ship_Mode
    LEFT JOIN dw01.silver.t_dates dt_order
        ON TO_DATE(o.Order_Date, 'dd-MM-yyyy') = dt_order.date
    LEFT JOIN dw01.silver.t_dates dt_ship
        ON TO_DATE(o.Ship_Date, 'dd-MM-yyyy') = dt_ship.date
)
SELECT *
FROM main;


In [0]:
%sql
USE dw01.gold;

WITH unique_customers AS (
    SELECT *
    FROM (
        SELECT *,
               ROW_NUMBER() OVER (
                   PARTITION BY Customer_Name, City, State, Customer_Type, Account_Manager
                   ORDER BY customer_id
               ) AS rn
        FROM dw01.silver.t_customers
    ) t
    WHERE rn = 1
),
unique_products AS (
    SELECT *
    FROM (
        SELECT *,
               ROW_NUMBER() OVER (
                   PARTITION BY Product_Name, Product_Category, Product_Container
                   ORDER BY product_id
               ) AS rn
        FROM dw01.silver.t_products
    ) t
    WHERE rn = 1
),
unique_shipping AS (
    SELECT *
    FROM (
        SELECT *,
               ROW_NUMBER() OVER (
                   PARTITION BY Ship_Mode
                   ORDER BY shipping_id
               ) AS rn
        FROM dw01.silver.t_shippingdetails
    ) t
    WHERE rn = 1
),
staging AS (
    SELECT
        o.order_id,
        o.Order_No,
        c.customer_id,
        p.product_id,
        s.shipping_id,
        dt_order.date_id AS order_date_id,
        dt_ship.date_id AS ship_date_id,
        o.Customer_Name,
        o.Address,
        o.City,
        o.State,
        o.Customer_Type,
        o.Account_Manager,
        o.Order_Priority,
        o.Product_Name,
        p.Product_Category,
        p.Product_Container,
        o.Ship_Mode,
        o.Ship_Date,
        o.Cost_Price,
        o.Retail_Price,
        o.Profit_Margin,
        o.Order_Quantity,
        o.Sub_Total,
        o.Discount_Amount,
        o.Order_Total,
        o.Shipping_Cost,
        o.Total
    FROM dw01.silver.t_orders o
    LEFT JOIN unique_customers c
        ON o.Customer_Name = c.Customer_Name
       AND o.City = c.City
       AND o.State = c.State
       AND o.Customer_Type = c.Customer_Type
       AND o.Account_Manager = c.Account_Manager
    LEFT JOIN unique_products p
        ON o.Product_Name = p.Product_Name
       AND o.Product_Category = p.Product_Category
       AND o.Product_Container = p.Product_Container
    LEFT JOIN unique_shipping s
        ON o.Ship_Mode = s.Ship_Mode
    LEFT JOIN dw01.silver.t_dates dt_order
        ON TO_DATE(o.Order_Date, 'dd-MM-yyyy') = dt_order.date
    LEFT JOIN dw01.silver.t_dates dt_ship
        ON TO_DATE(o.Ship_Date, 'dd-MM-yyyy') = dt_ship.date
)

-- Append-only incremental insert
INSERT INTO t_orders (
    order_id, Order_No, customer_id, product_id, shipping_id,
    order_date_id, ship_date_id, Customer_Name, Address, City, State,
    Customer_Type, Account_Manager, Order_Priority, Product_Name,
    Product_Category, Product_Container, Ship_Mode, Ship_Date,
    Cost_Price, Retail_Price, Profit_Margin, Order_Quantity, Sub_Total,
    Discount_Amount, Order_Total, Shipping_Cost, Total
)
SELECT *
FROM staging s
WHERE NOT EXISTS (
    SELECT 1
    FROM t_orders t
    WHERE t.order_id = s.order_id
);


In [0]:
%sql
with main as (
SELECT
    o.order_id,
    o.Order_No,
    c.customer_id,
    p.product_id,
    s.shipping_id,
    dt_order.date_id AS order_date_id,
    dt_ship.date_id AS ship_date_id,
    o.Customer_Name,
    o.Address,
    o.City,
    o.State,
    o.Customer_Type,
    o.Account_Manager,
    o.Order_Priority,
    o.Product_Name,
    p.Product_Category,
    p.Product_Container,
    o.Ship_Mode,
    o.Ship_Date,
    o.Cost_Price,
    o.Retail_Price,
    o.Profit_Margin,
    o.Order_Quantity,
    o.Sub_Total,
    o.Discount_Amount,
    o.Order_Total,
    o.Shipping_Cost,
    o.Total
FROM dw01.silver.t_orders o
LEFT JOIN dw01.silver.t_customers c
    ON o.Customer_Name = c.Customer_Name
   AND o.City = c.City
   AND o.State = c.State
   AND o.Customer_Type = c.Customer_Type
   AND o.Account_Manager = c.Account_Manager
LEFT JOIN dw01.silver.t_products p
    ON o.Product_Name = p.Product_Name
   AND o.Product_Category = p.Product_Category
   AND o.Product_Container = p.Product_Container
LEFT JOIN dw01.silver.t_shippingdetails s
    ON o.Ship_Mode = s.Ship_Mode
-- Join to date dimension for Order_Date
LEFT JOIN dw01.silver.t_dates dt_order
    ON TO_DATE(o.Order_Date, 'dd-MM-yyyy') = dt_order.date
-- Join to date dimension for Ship_Date
LEFT JOIN dw01.silver.t_dates dt_ship
    ON TO_DATE(o.Ship_Date, 'dd-MM-yyyy') = dt_ship.date
), subquery as 
(
SELECT
    order_id,
    customer_id,
    product_id,
    shipping_id,
    order_date_id,
    ship_date_id,
    Order_No,
    Cost_Price,
    Retail_Price,
    Profit_Margin,
    Order_Quantity,
    Sub_Total,
    Discount_Amount,
    Order_Total,
    Shipping_Cost,
    Total from main)

select * from subquery;


In [0]:
%sql
describe dw01.silver.t_customers

In [0]:
%sql
desc dw01.silver.t_orders

In [0]:
%sql
select * from dw01.silver.t_dates
limit 1

In [0]:
%sql
describe dw01.silver.t_dates